In this notebook, we focus on predicting the match result. 

In [1]:
import pandas as pd
import numpy as np

## Preparation

In [2]:
data = pd.read_csv('/Users/tliu/Desktop/Erdos Project/3_Player_Data_Generation/match_data_50_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p1_frames_played_3_years,p1_frames_won_3_years,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage
0,Cao Yupeng,Jiang Jun,9,1411,1044,0.918016,0.714532,341,179,2302,...,446,242,32,17,32,17,5,0,0.0,1.000000
1,Siripaporn Nuanthakhamjan,Zhou Yuelong,9,965,1384,0.056881,0.259705,3,0,23,...,23,4,245,118,808,430,2,5,1.0,0.285714
2,Wu Yize,Allan Taylor,9,1342,1237,0.656987,0.565251,78,37,552,...,404,215,101,49,336,147,5,3,0.0,0.625000
3,Ben Woollaston,Oliver Brown,9,1412,1146,0.845318,0.660383,803,454,5053,...,534,282,91,35,235,97,5,2,0.0,0.714286
4,Andres Petrov,Mark Williams,9,1045,1611,0.017765,0.195447,51,18,304,...,167,61,281,170,1112,649,2,5,1.0,0.285714


In [3]:
#Add more features
data['p1_frames_win_rate'] = data['p1_frames_won']/ data['p1_frames_played']
data['p2_frames_win_rate'] = data['p2_frames_won']/ data['p2_frames_played']

data['p1_matches_win_rate'] = data['p1_matches_won']/ data['p1_matches_played']
data['p2_matches_win_rate'] = data['p2_matches_won']/ data['p2_matches_played']

#p1_matches_win_rate and p2_matches_win_rate both have missing values
#We will fill them with 0.5
data.fillna(0.5, inplace = True)

#Drop players' names from the data since we won't consider strings as our features for modeling
data = data.drop(['player1', 'player2'], axis = 1)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5058 entries, 0 to 5057
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   best_of                   5058 non-null   int64  
 1   player1_elo               5058 non-null   int64  
 2   player2_elo               5058 non-null   int64  
 3   elo_match_win_rate        5058 non-null   float64
 4   elo_frame_win_rate        5058 non-null   float64
 5   p1_matches_played         5058 non-null   int64  
 6   p1_matches_won            5058 non-null   int64  
 7   p1_frames_played          5058 non-null   int64  
 8   p1_frames_won             5058 non-null   int64  
 9   p2_matches_played         5058 non-null   int64  
 10  p2_matches_won            5058 non-null   int64  
 11  p2_frames_played          5058 non-null   int64  
 12  p2_frames_won             5058 non-null   int64  
 13  p1_frames_played_1_year   5058 non-null   int64  
 14  p1_frame

In [5]:
data.columns

Index(['best_of', 'player1_elo', 'player2_elo', 'elo_match_win_rate',
       'elo_frame_win_rate', 'p1_matches_played', 'p1_matches_won',
       'p1_frames_played', 'p1_frames_won', 'p2_matches_played',
       'p2_matches_won', 'p2_frames_played', 'p2_frames_won',
       'p1_frames_played_1_year', 'p1_frames_won_1_year',
       'p1_frames_played_3_years', 'p1_frames_won_3_years',
       'p2_frames_played_1_year', 'p2_frames_won_1_year',
       'p2_frames_played_3_years', 'p2_frames_won_3_years', 'score1', 'score2',
       'match_result', 'win_percentage', 'p1_frames_win_rate',
       'p2_frames_win_rate', 'p1_matches_win_rate', 'p2_matches_win_rate'],
      dtype='object')

In [6]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle=False)


In [7]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings) and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2'], axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage','score1', 'score2'], axis = 1)

## Performance Metrics
Because the two classes in the target are symmetric (swapping player1 and player2 will exchange positive and negative but still represents the same match), we won't consider metrics such as presision, specificity and sensitivity since they are the same as accuracy score. We will only consider accuracy score.

In [8]:
#Import metrics
from sklearn.metrics import accuracy_score

In [9]:
#Create dictionaries to store the metrics.
accuracy_scores = {}

#Create a list to store all the models we consider.
models = {}

In [10]:
def print_avg_cv_metrics(model, model_name):
    """
    Given a model, computes and stores accuracy scores on the test set.
    """

    #Create an empty array to store the scores.
    print('Currently working on ' + model_name + '.')

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    print('The accuracy score of ' + model_name + ' is:', score)
    
    #Record the scores
    accuracy_scores[model_name] = score

    models[model_name] = model


## Model 0: Prediction by elo rating

This model predicts the winner to the player with higher elo rating. If two players have the same elo ratings, it predicts player1 to win. 

In [11]:
#Performance on the training set
pred_by_elo_train = X_train['player1_elo'] < X_train['player2_elo']
score1 = accuracy_score(y_train, pred_by_elo_train)

print('The accuracy score of prediction by elo rating on the training set is:', score1)

#Performance on the test set
pred_by_elo = X_test['player1_elo'] < X_test['player2_elo']
score2 = accuracy_score(y_test, pred_by_elo)

print('The accuracy score of prediction by elo rating is:', score2)

#Record the scores
accuracy_scores['Predict by elo'] = score2

The accuracy score of prediction by elo rating on the training set is: 0.6777063766683143
The accuracy score of prediction by elo rating is: 0.6462450592885376


## Model 1: Logistic Regression with PCA

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

In [13]:
#Get the number of features in X_train
print(len(X_train.columns))

25


In [14]:
scaler = StandardScaler()
pca = PCA()
log_reg = LogisticRegression(max_iter=10000)

log_reg_with_pca = Pipeline([('scale', scaler),
                    ('pca', pca),
                    ('logistic', log_reg)])

param_grid = {
    "pca__n_components": list(range(5, 26, 5)),
    "logistic__C": np.logspace(-4, 4, 8),
}


grid_search = GridSearchCV(log_reg_with_pca,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search.fit(X_train, y_train.values)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('pca', PCA()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.00000000e-04, 1.38949549e-03, 1.93069773e-02, 2.68269580e-01,
       3.72759372e+00, 5.17947468e+01, 7.19685673e+02, 1.00000000e+04]),
                         'pca__n_components': [5, 10, 15, 20, 25]},
             scoring='accuracy')

Notice that for linear models, pca__n_components = number_of_features is the same as modeling without PCA. 

In [15]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'logistic__C': np.float64(0.2682695795279725), 'pca__n_components': 20}
0.6836347266095928


In [16]:
model1 = grid_search.best_estimator_
print_avg_cv_metrics(model1, 'Logistic Regression with PCA')

Currently working on Logistic Regression with PCA.
The accuracy score of Logistic Regression with PCA is: 0.6403162055335968


## Model 2: Random Forest

In [17]:
from sklearn.ensemble import RandomForestClassifier

In [18]:
random_forest = RandomForestClassifier()

param_grid = {
    "max_depth": [3, 4, 5, 6, 7, 8],
    "n_estimators": np.linspace(100, 800, 8).astype(int)
}


grid_search2 = GridSearchCV(random_forest,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search2.fit(X_train, y_train)

GridSearchCV(cv=5, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [3, 4, 5, 6, 7, 8],
                         'n_estimators': array([100, 200, 300, 400, 500, 600, 700, 800])},
             scoring='accuracy')

In [19]:
print(grid_search2.best_params_)
print(grid_search2.best_score_)

{'max_depth': 4, 'n_estimators': np.int64(500)}
0.6836325901509255


In [20]:
model2 = grid_search2.best_estimator_
print_avg_cv_metrics(model2, 'Random Forest')

Currently working on Random Forest.
The accuracy score of Random Forest is: 0.6600790513833992


In [21]:
print(accuracy_scores)

{'Predict by elo': 0.6462450592885376, 'Logistic Regression with PCA': 0.6403162055335968, 'Random Forest': 0.6600790513833992}
